# Specimen 03 — Task Queue & Supervisor

Goal: replace a fixed batch of tasks handed out all at once with a queue that workers pull from as they become free, plus a supervisor that retries failed tasks and aggregates results as they actually complete -- not in the order they were submitted.

In [1]:
import os
import json
import time
import asyncio
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.AsyncAnthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

# Caps how many requests are in flight at once, regardless of how many coroutines are scheduled --
# asyncio.gather alone fires every call simultaneously, which is the fastest way to get rate-limited.
_concurrency_limit = asyncio.Semaphore(5)

async def call_model(messages, tools=None, max_tokens=1200, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    async with _concurrency_limit:
        return await client.messages.create(**kwargs)


`call_model` is now `async` and built on `AsyncAnthropic` -- `asyncio.gather` needs real awaitable coroutines to get genuine concurrency; wrapping the sync client wouldn't actually overlap requests. A `Semaphore` caps in-flight requests at 5, since `gather` alone will fire every call at once. Everything else carries forward from Phase 3-5: `thinking` disabled by default, schema-enforced JSON handoffs via `output_schema`, and a default `max_tokens` of 1200 -- Phase 5 found 800 too tight for planner/researcher-style structured output, where the model kept writing past the budget and breaking the JSON mid-generation. One more Phase 5 finding worth remembering here: `output_config`'s JSON schema does not support `maxItems` on arrays -- bound response length through the prompt (ask for an exact count, not "at most"), not the schema.

## 1. Build an `asyncio.Queue` seeded with a fixed set of tasks

Each item is one unit of work a worker can pull and process independently.

In [2]:
task_list = [
    "Summarize the plot of a story about a lighthouse keeper who discovers a message in a bottle, in one sentence.",
    "Explain in one sentence why the sky appears blue.",
    "Give one interesting fact about octopuses in one sentence.",
    "Explain in one sentence what a black hole is.",
    "Give one fact about the history of chess in one sentence.",
    "Explain in one sentence how vaccines work.",
    "Give one fact about the Great Barrier Reef in one sentence.",
    "Explain in one sentence why leaves change color in autumn.",
]

task_queue = asyncio.Queue()
for i, task in enumerate(task_list):
    await task_queue.put((i, task))

print(f"Queue seeded with {task_queue.qsize()} tasks")

Queue seeded with 8 tasks


## 2. Define worker coroutines that loop against the queue

Pull a task, process it (call the model), record the result, repeat until the queue is empty. This is a pull model, not a fixed 1:1 assignment.

In [3]:
async def queue_worker(worker_id, queue, results):
    while True:
        try:
            task_id, task_text = queue.get_nowait()
        except asyncio.QueueEmpty:
            return
        response = await call_model(messages=[{"role": "user", "content": task_text}], max_tokens=150)
        answer = ''.join(b.text for b in response.content if b.type == 'text')
        results.append({"task_id": task_id, "worker_id": worker_id, "answer": answer})
        queue.task_done()

results_single = []
await asyncio.gather(queue_worker(0, task_queue, results_single))
print(f"Single worker processed {len(results_single)} tasks")
for r in sorted(results_single, key=lambda r: r["task_id"]):
    print(f"  [{r['task_id']}] {r['answer']}")

Single worker processed 8 tasks
  [0] An aging lighthouse keeper, long accustomed to the solitude of his rocky outpost, finds a weathered bottle washed up at the tideline containing a letter—whether a stranger's confession, a plea for rescue, or a note he himself cast into the sea decades earlier—and its words compel him to reckon with the life he abandoned on shore, ultimately drawing him back toward the world he thought he'd left behind for good.
  [1] The sky appears blue because air molecules scatter shorter-wavelength (blue) sunlight far more strongly than longer-wavelength (red) light — an effect called Rayleigh scattering — so blue light gets redirected all across the sky toward our eyes.
  [2] Octopuses have three hearts—two pump blood to the gills while the third circulates it to the rest of the body, and that third heart stops beating when they swim, which is part of why they prefer crawling.
  [3] A black hole is a region of space where gravity is so intense — typically from

## 3. Run several workers concurrently and confirm dynamic load balancing

Launch more than one worker coroutine against the same queue. Confirm a worker that finishes early picks up another task immediately, rather than sitting idle while a slower worker is still on its first one -- log which worker handled which task to make this visible.

In [4]:
task_queue2 = asyncio.Queue()
for i, task in enumerate(task_list):
    await task_queue2.put((i, task))

results_multi = []
start = time.perf_counter()
await asyncio.gather(*[queue_worker(w, task_queue2, results_multi) for w in range(3)])
elapsed = time.perf_counter() - start

print(f"3 workers processed {len(results_multi)} tasks in {elapsed:.2f}s")
counts = {}
for r in results_multi:
    counts[r["worker_id"]] = counts.get(r["worker_id"], 0) + 1
print(f"Tasks per worker: {counts}")
print("\nCompletion order (task_id -> worker_id), not submission order:")
for r in results_multi:
    print(f"  task {r['task_id']} -> worker {r['worker_id']}")

3 workers processed 8 tasks in 11.53s
Tasks per worker: {2: 4, 0: 2, 1: 2}

Completion order (task_id -> worker_id), not submission order:
  task 2 -> worker 2
  task 0 -> worker 0
  task 1 -> worker 1
  task 3 -> worker 2
  task 6 -> worker 2
  task 5 -> worker 1
  task 4 -> worker 0
  task 7 -> worker 2


## 4. Add a supervisor that retries a failed task once

On an exception (or a deliberately injected bad result), the supervisor re-queues the task once before giving up and recording a permanent failure.

In [5]:
async def queue_worker_supervised(worker_id, queue, results, max_retries=1):
    while True:
        try:
            task_id, task_text, attempt = queue.get_nowait()
        except asyncio.QueueEmpty:
            return
        try:
            if "POISON" in task_text:
                raise RuntimeError(f"Simulated failure on task {task_id}")
            response = await call_model(messages=[{"role": "user", "content": task_text}], max_tokens=150)
            answer = ''.join(b.text for b in response.content if b.type == 'text')
            results.append({"task_id": task_id, "worker_id": worker_id, "status": "ok", "answer": answer, "attempt": attempt})
        except Exception as e:
            if attempt < max_retries:
                print(f"[worker {worker_id}] task {task_id} failed (attempt {attempt}), retrying: {e}")
                await queue.put((task_id, task_text, attempt + 1))
            else:
                print(f"[worker {worker_id}] task {task_id} failed permanently after {attempt + 1} attempts: {e}")
                results.append({"task_id": task_id, "worker_id": worker_id, "status": "failed", "answer": None, "attempt": attempt})
        finally:
            queue.task_done()

poison_tasks = task_list + ["POISON -- this task always fails"]
task_queue3 = asyncio.Queue()
for i, task in enumerate(poison_tasks):
    await task_queue3.put((i, task, 0))

results_supervised = []
await asyncio.gather(*[queue_worker_supervised(w, task_queue3, results_supervised) for w in range(3)])

succeeded = [r for r in results_supervised if r["status"] == "ok"]
failed = [r for r in results_supervised if r["status"] == "failed"]
print(f"\nSucceeded: {len(succeeded)}, Permanently failed: {len(failed)}")
for r in failed:
    print(f"  Task {r['task_id']} failed after {r['attempt'] + 1} attempt(s)")

[worker 0] task 8 failed (attempt 0), retrying: Simulated failure on task 8
[worker 0] task 8 failed permanently after 2 attempts: Simulated failure on task 8



Succeeded: 8, Permanently failed: 1
  Task 8 failed after 2 attempt(s)


## 5. Aggregate results as they complete, not as they were submitted

Log completion order alongside original submission order, and show they can differ -- this is what 'aggregating results that arrive out of order' actually looks like, not just a claim in the roadmap notes.

In [6]:
submission_order = list(range(len(poison_tasks)))
completion_order = [r["task_id"] for r in results_supervised if r["status"] == "ok"]
print(f"Submission order: {submission_order[:-1]}")
print(f"Completion order: {completion_order}")
print(f"Same order: {submission_order[:len(completion_order)] == completion_order}")

Submission order: [0, 1, 2, 3, 4, 5, 6, 7]
Completion order: [2, 1, 0, 3, 4, 5, 6, 7]
Same order: False


## 6. Add a per-task timeout

Wrap each task's model call in `asyncio.wait_for` with a timeout. Confirm one artificially slow/stuck task times out and gets handled (retried or failed) without stalling the other workers still pulling from the queue.

In [7]:
async def slow_or_fast_task(task_text, timeout):
    if "SLOWPOKE" in task_text:
        await asyncio.sleep(timeout + 5)  # deliberately exceeds the timeout
        return "should never get here"
    response = await call_model(messages=[{"role": "user", "content": task_text}], max_tokens=150)
    return ''.join(b.text for b in response.content if b.type == 'text')

async def queue_worker_timeout(worker_id, queue, results, timeout=8, max_retries=1):
    while True:
        try:
            task_id, task_text, attempt = queue.get_nowait()
        except asyncio.QueueEmpty:
            return
        try:
            answer = await asyncio.wait_for(slow_or_fast_task(task_text, timeout), timeout=timeout)
            results.append({"task_id": task_id, "worker_id": worker_id, "status": "ok", "answer": answer})
        except asyncio.TimeoutError:
            if attempt < max_retries:
                print(f"[worker {worker_id}] task {task_id} timed out (attempt {attempt}), retrying")
                await queue.put((task_id, task_text, attempt + 1))
            else:
                print(f"[worker {worker_id}] task {task_id} timed out permanently after {attempt + 1} attempts")
                results.append({"task_id": task_id, "worker_id": worker_id, "status": "timeout"})
        finally:
            queue.task_done()

timeout_tasks = task_list[:4] + ["SLOWPOKE -- this task takes forever"]
task_queue4 = asyncio.Queue()
for i, t in enumerate(timeout_tasks):
    await task_queue4.put((i, t, 0))

results_timeout = []
start = time.perf_counter()
await asyncio.gather(*[queue_worker_timeout(w, task_queue4, results_timeout, timeout=8) for w in range(3)])
elapsed = time.perf_counter() - start

print(f"\nTotal wall-clock: {elapsed:.1f}s")
for r in sorted(results_timeout, key=lambda r: r["task_id"]):
    print(f"  task {r['task_id']}: {r['status']}")

[worker 1] task 4 timed out (attempt 0), retrying


[worker 1] task 4 timed out permanently after 2 attempts

Total wall-clock: 19.5s
  task 0: ok
  task 1: ok
  task 2: ok
  task 3: ok
  task 4: timeout
